# Polymarket LP Rewards Miner
===============================

Goal: Earn liquidiy rewards by always maintaining qualifying resting orders on BOTH sides of the market.  NOT a directional trading strategy.

How YES/NO tokens work on Polymarket:
- Each market has TWO separate token IDs: one for YES, one for NO
- YES price + No price always sum to ~1.00 (e.g. YES=0.60, NO=0.40)
- We place a BUY order on each token at a discount from their respective mids
- "Buying No at 0.38" is economically the same as "selling YES at 0.62"
- Both are resting BID orders in the book -> both qualify for LP rewards

Strategy:
- Place BUY and (mid-max_spread) and SELL at (mid + max_spread)
- Use exactly the minimum qualifying share size
- If any order gets filled (position taken), immediately flatten at market
Continously watch for mid price drift and requote when needed

Requirements:
```
pip install py-clob-client python-dotenv websockets
```

Setup (.env file)

In [1]:
import os
import time
import logging
import requests
from datetime import datetime, timezone
from decimal import Decimal, ROUND_DOWN
from dotenv import load_dotenv
 
from py_clob_client.client import ClobClient
from py_clob_client.clob_types import (
    ApiCreds,
    OrderArgs,
    MarketOrderArgs,
    OpenOrderParams,
    BalanceAllowanceParams,
    AssetType,
)
from py_clob_client.constants import POLYGON

In [2]:
load_dotenv()

True

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)

## MARKET CONFIG

In [4]:
MARKETS = [
    # {
    #     "label": "Michael B. Jordan Best Actor",
    #     "slug":  "will-michael-b-jordan-win-best-actor-at-the-98th-academy-awards",
    # },
    {
        "label": "Seoul Temp",
        "slug":  "highest-temperature-in-seoul-on-april-1-2026-13c",
    },
]

## BOT SETTINGS

In [5]:
# how far mid must drift before we cancel & requote (in price points)
# Requote if mid moves more than this since last quote
REQUOTE_THRESHOLD = Decimal("0.005")

# How often to poll for fills and mid price changes (seconds)
POLL_INTERVAL = 10

# Safety buffer: place sightly inside max_sprad so floating point doesn't accidentally push us outside the qualifying range
SPREAD_BUFFER = Decimal("0.001")

# Price tick size on Polymarket
TICK = Decimal("0.001")

SIGNATURE_TYPE = int(os.getenv("SIGNATURE_TYPE", "1"))
FLAT_EPS = Decimal("0.000001")

## CLIENT

In [6]:
def build_client() -> ClobClient:
    creds = ApiCreds(
        api_key=os.environ["POLYMARKET_API_KEY"],
        api_secret=os.environ["POLYMARKET_API_SECRET"],
        api_passphrase=os.environ["POLYMARKET_API_PASSPHRASE"],
    )
    return ClobClient(
        host="https://clob.polymarket.com",
        chain_id=POLYGON,
        key=os.environ["POLYMARKET_PRIVATE_KEY"],
        creds=creds,
        signature_type=SIGNATURE_TYPE,
        funder=os.environ["POLYMARKET_FUNDER"],
    )

## REWARD PARAMS - fetched live from Polymarket gamma API

In [7]:
def fetch_market_config(slug: str, label: str) -> dict | None:
    """
    Fetch all market parameters from the Polymarket gamma API using just the slug.
    Returns a fully populated market config dict, or None on failure.
 
    Fetches:
      - yes_token_id, no_token_id
      - end_date
      - max_spread (converts from % to decimal, e.g. 4.5 -> 0.045)
      - min_size
      - daily_rate
    """
    try:
        resp = requests.get(
            "https://gamma-api.polymarket.com/markets",
            params={"slug": slug},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json()
        if not results:
            log.error(f"[{label}] No market found for slug: {slug}")
            return None
        data = results[0]
 
        # Parse token IDs
        import json
        token_ids = json.loads(data.get("clobTokenIds", "[]"))
        if len(token_ids) < 2:
            log.error(f"[{label}] Expected 2 token IDs, got: {token_ids}")
            return None
 
        # Determine which is YES and which is NO from outcomes
        outcomes = json.loads(data.get("outcomes", '["Yes", "No"]'))
        if outcomes[0].lower() == "yes":
            yes_token_id, no_token_id = token_ids[0], token_ids[1]
        else:
            yes_token_id, no_token_id = token_ids[1], token_ids[0]
 
        # Parse end date
        end_date_str = data.get("endDate", "")
        end_date = datetime.fromisoformat(end_date_str.replace("Z", "+00:00"))
 
        # Reward params
        max_spread_pct = Decimal(str(data.get("rewardsMaxSpread", 5)))
        max_spread     = max_spread_pct / Decimal("100")
        min_size       = Decimal(str(data.get("rewardsMinSize", 50)))
        rewards        = data.get("clobRewards", [{}])[0]
        daily_rate     = Decimal(str(rewards.get("rewardsDailyRate", 0)))
 
        log.info(
            f"[{label}] Market config fetched:"
            f"  YES token: {yes_token_id[:16]}..."
            f"  NO  token: {no_token_id[:16]}..."
            f"  End date:  {end_date.date()}"
            f"  Max spread: {max_spread} ({max_spread_pct}%) | "
            f"Min size: {min_size} | Daily rate: ${daily_rate}"
        )
 
        return {
            "label":        label,
            "slug":         slug,
            "yes_token_id": yes_token_id,
            "no_token_id":  no_token_id,
            "end_date":     end_date,
            "max_spread":   max_spread,
            "min_size":     min_size,
            "daily_rate":   daily_rate,
        }
 
    except Exception as e:
        log.error(f"[{label}] fetch_market_config failed: {e}")
        return None

## Market Data Helpers

In [8]:
def get_midpoint(client: ClobClient, token_id: str) -> Decimal | None:
    """Fetch best bid/ask for a token and return the midpoint."""
    try:
        ob = client.get_order_book(token_id)
        if not ob.bids or not ob.asks:
            log.warning(f"[{token_id[:8]}...] One-sided or empty book.")
            return None
        bids = sorted(ob.bids, key=lambda x: float(x.price), reverse=True)
        asks = sorted(ob.asks, key=lambda x: float(x.price), reverse=False)
        best_bid = Decimal(str(bids[0].price))
        best_ask = Decimal(str(asks[0].price))
        return ((best_bid + best_ask) / 2).quantize(TICK)
    except Exception as e:
        log.error(f"get_midpoint({token_id[:8]}) error: {e}")
        return None


def get_open_order_ids(client: ClobClient, token_id: str) -> set | None:
    """
    Returns set of open order IDs for this token.
    Returns None (not empty set) on API error so callers can distinguish
    'no orders' from 'API failed' and avoid false fill detection.
    """
    try:
        orders = client.get_orders(OpenOrderParams(asset_id=token_id))
        return {o["id"] for o in (orders or [])}
    except Exception as e:
        log.error(f"get_open_order_ids error: {e}")
        return None


def cancel_order_ids(client: ClobClient, order_ids: list) -> None:
    if not order_ids:
        return
    try:
        client.cancel_orders(order_ids)
        log.info(f"Cancelled {len(order_ids)} order(s): {order_ids}")
    except Exception as e:
        log.error(f"cancel_orders error: {e}")


def get_token_position(client: ClobClient, token_id: str) -> Decimal:
    """
    Return shares held for this token.
    Primary: balance-allowance API (accurate).
    Fallback: net from trade history.
    """
    try:
        params = BalanceAllowanceParams(
            asset_type=AssetType.CONDITIONAL,
            token_id=token_id,
            signature_type=SIGNATURE_TYPE,
        )
        resp = client.get_balance_allowance(params=params)
        raw  = Decimal(str(resp.get("balance", "0")))
        return raw / Decimal("1000000")   # 6 decimal USDC units → shares
    except Exception as e:
        log.warning(f"get_token_position primary failed: {e} — trying trade history fallback.")
        try:
            from py_clob_client.clob_types import TradeParams
            trades = client.get_trades(TradeParams(asset_id=token_id)) or []
            net = Decimal("0")
            for t in trades:
                size = Decimal(str(t.get("size", 0)))
                net += size if t.get("side", "").upper() == "BUY" else -size
            log.info(f"get_token_position fallback result: {net:.4f} shares")
            return net
        except Exception as e2:
            log.error(f"get_token_position fallback also failed: {e2}")
            return Decimal("0")

def get_usdc_balance(client: ClobClient) -> Decimal:
    """Return available USDC balance in the trading account.
    Balance is returned in raw USDC units (6 decimals)
    """
    try:
        resp = client.get_balance_allowance(params=BalanceAllowanceParams(asset_type=AssetType.COLLATERAL))
        raw = Decimal(str(resp.get("balance", "0")))
        return raw / Decimal("1000000")   # convert from 6-decimal units to USDC
    except Exception as e:
        log.error(f"get_usdc_balance error: {e}")
        return Decimal("0")



## Order Actions

In [ ]:
def place_buy_limit(
    client: ClobClient,
    token_id: str,
    price: Decimal,
    size: Decimal,
    label: str,
) -> str | None:
    """Place a resting GTC BUY limit order. Returns order ID or None."""
    price = max(Decimal("0.01"), min(Decimal("0.99"), price))
    try:
        resp = client.create_and_post_order(OrderArgs(
            token_id=token_id,
            price=float(price),
            size=float(size),
            side="BUY",
        ))
        oid = resp.get("orderID") or resp.get("id")
        log.info(f"[{label}] BUY limit @ {price:.4f} x {size} shares → {oid}")
        return oid
    except Exception as e:
        log.error(f"[{label}] place_buy_limit failed: {e}")
        return None


def sell_at_market(
    client: ClobClient,
    token_id: str,
    size: Decimal,
    label: str,
) -> bool:
    """Sell shares immediately at market price to go flat. Returns True on success."""
    log.warning(f"[{label}] 🔴 Flattening — selling {size:.4f} shares at market...")
    try:
        # Post an aggressive SELL limit at the lowest valid price (0.01)
        # This will match against any resting bids and clear immediately
        resp = client.create_and_post_order(MarketOrderArgs(
            token_id=token_id,          # sweep the book — will fill at best available bid
            size=float(size),
            side="SELL",
        ))
        order_id = resp.get("orderID") or resp.get("id")
        status   = resp.get("status", "unknown")
        log.info(f"[{label}] Flatten order posted → {order_id} (status={status})")
        return True

    except Exception as e:
        err = str(e).lower()
        if any(w in err for w in ["balance", "insufficient", "funds"]):
            log.error(f"[{label}] ⚠️  Flatten failed — insufficient balance. Will retry next cycle.")
        else:
            log.error(f"[{label}] ⚠️  Flatten failed: {e}. MANUAL INTERVENTION MAY BE NEEDED.")
        return False
    
    
def sell_all_positions(client: ClobClient, token_ids: list[str]) -> None:
    """
    Check every token for a held position and sell all at market immediately.
    Retries each token until confirmed flat or max retries exceeded.
    
    Usage:
        sell_all_positions(auth_client, [YES_TOKEN_ID, NO_TOKEN_ID, ...])
    """
    FLATTEN_MAX_RETRIES    = 10
    FLATTEN_RETRY_INTERVAL = 5  # seconds between retries

    for token_id in token_ids:
        log.info(f"Checking position for token {token_id[:16]}...")

        for attempt in range(1, FLATTEN_MAX_RETRIES + 1):
            # Check current position
            try:
                from py_clob_client.clob_types import BalanceAllowanceParams, AssetType
                resp = client.get_balance_allowance(params=BalanceAllowanceParams(
                    asset_type=AssetType.CONDITIONAL,
                    token_id=token_id,
                    signature_type=SIGNATURE_TYPE,
                ))
                pos = Decimal(str(resp.get("balance", "0"))) / Decimal("1000000")
            except Exception as e:
                log.error(f"  Could not fetch position: {e}")
                break

            if pos <= Decimal("0.01"):
                log.info(f"  ✓ Flat (position={pos:.4f})")
                break

            log.warning(f"  Attempt {attempt}/{FLATTEN_MAX_RETRIES} — holding {pos:.4f} shares. Selling...")

            try:
                resp = client.create_and_post_order(OrderArgs(
                    token_id=token_id,
                    price=0.01,       # aggressive price — sweeps all resting bids
                    size=float(pos),
                    side="SELL",
                ))
                log.info(f"  Sell posted → {resp.get('orderID') or resp.get('id')} (status={resp.get('status')})")
            except Exception as e:
                log.error(f"  Sell failed: {e}")

            time.sleep(FLATTEN_RETRY_INTERVAL)

        else:
            # Loop exhausted without breaking — still not flat
            log.error(f"  ❌ Could not flatten {token_id[:16]}... after {FLATTEN_MAX_RETRIES} attempts. MANUAL ACTION NEEDED.")


## Pre-market State

In [ ]:
class MarketMaker:
    """
    Manages LP reward quoting for one prediction market.
 
    Maintains exactly two resting BUY orders:
      - BUY YES at (yes_mid - effective_spread)
      - BUY NO  at (no_mid  - effective_spread)
 
    Reward params (max_spread, min_size) are fetched from the API on startup
    and refreshed every PARAM_REFRESH_INTERVAL cycles.
    """
 
    PARAM_REFRESH_INTERVAL = 60   # refresh reward params every N cycles (~10 min)
 
    def __init__(self, cfg: dict):
        self.label        = cfg["label"]
        self.slug         = cfg["slug"]
        self.yes_token_id = cfg["yes_token_id"]
        self.no_token_id  = cfg["no_token_id"]
        self.end_date     = cfg["end_date"]
        self.max_spread   = cfg["max_spread"]
        self.min_size     = cfg["min_size"]
        self.daily_rate   = cfg["daily_rate"]
 
        self.yes_order_id:   str | None = None
        self.no_order_id:    str | None = None
        self.quoted_yes_mid: Decimal | None = None
        self._cycle_count:   int = 0
 
    # ── Reward param refresh ───────────────────────────────
 
    def refresh_params(self) -> None:
        """Re-fetch all market params from API in case rewards program changed."""
        updated = fetch_market_config(self.slug, self.label)
        if updated:
            self.max_spread = updated["max_spread"]
            self.max_spread = self.max_spread - Decimal(0.005)
            self.min_size   = updated["min_size"]
            self.daily_rate = updated["daily_rate"]
            log.info(
                f"[{self.label}] Params refreshed — "
                f"max_spread={self.max_spread} | min_size={self.min_size} | "
                f"daily_rate=${self.daily_rate}"
            )
 
    # ── Price helpers ──────────────────────────────────────
 
    def _buy_price(self, mid: Decimal) -> Decimal:
        """Widest qualifying BUY price = mid - (max_spread - buffer), floored to tick."""
        effective = self.max_spread - SPREAD_BUFFER
        return max(Decimal("0.01"), (mid - effective).quantize(TICK, rounding=ROUND_DOWN))
 
    # ── State checks ───────────────────────────────────────
 
    def is_expired(self) -> bool:
        return datetime.now(timezone.utc) >= self.end_date
 
    def mid_has_drifted(self, new_yes_mid: Decimal) -> bool:
        if self.quoted_yes_mid is None:
            return True
        log.info(f"Mid Drift: {abs(new_yes_mid - self.quoted_yes_mid)}")
        return abs(new_yes_mid - self.quoted_yes_mid) >= REQUOTE_THRESHOLD
 
    # ── Actions ────────────────────────────────────────────
 
    def cancel_all(self, client: ClobClient) -> None:
        ids = [oid for oid in [self.yes_order_id, self.no_order_id] if oid]
        cancel_order_ids(client, ids)
        self.yes_order_id = None
        self.no_order_id  = None
        
 
    def quote(self, client: ClobClient, yes_mid: Decimal) -> None:
        """Cancel stale orders, check balance, and post fresh BUY on both sides."""
        # Cancel first so reserved balance is freed before balance check
        self.cancel_all(client)
 
        no_mid  = (Decimal("1") - yes_mid).quantize(TICK)
        y_price = self._buy_price(yes_mid)
        n_price = self._buy_price(no_mid)
 
        yes_cost   = (y_price * self.min_size).quantize(Decimal("0.01"))
        no_cost    = (n_price * self.min_size).quantize(Decimal("0.01"))
        total_cost = yes_cost + no_cost
 
        balance = get_usdc_balance(client)
        log.info(f"[{self.label}] Balance: ${balance:.2f} | Required: ${total_cost:.2f}")
 
        if balance < total_cost:
            log.warning(
                f"[{self.label}] ⚠️  Insufficient balance — "
                f"need ${total_cost:.2f}, have ${balance:.2f}. Skipping quote."
            )
            return
 
        log.info(
            f"[{self.label}] Quoting — "
            f"BUY YES @ {y_price:.4f} | BUY NO @ {n_price:.4f} | "
            f"size={self.min_size} | cost=${total_cost:.2f}"
        )
 
        self.yes_order_id = place_buy_limit(
            client, self.yes_token_id, y_price, self.min_size, f"{self.label}/YES"
        )
        self.no_order_id = place_buy_limit(
            client, self.no_token_id, n_price, self.min_size, f"{self.label}/NO"
        )
        self.quoted_yes_mid = yes_mid
 
    def check_fills_and_flatten(self, client: ClobClient) -> bool:
        """
        Check if either order was filled and flatten immediately.
        Returns True if a fill was detected (triggers requote).
        If the open-orders API fails (returns None), conservatively skips
        fill detection to avoid false positives.
        """
        any_filled = False
 
        for order_id, token_id, side_label in [
            (self.yes_order_id, self.yes_token_id, "YES"),
            (self.no_order_id,  self.no_token_id,  "NO"),
        ]:
            if not order_id:
                continue
 
            open_ids = get_open_order_ids(client, token_id)
 
            if open_ids is None:
                # API error — skip fill check, don't assume filled
                log.warning(f"[{self.label}] Could not fetch {side_label} open orders — skipping fill check.")
                continue
 
            if order_id not in open_ids:
                log.warning(f"[{self.label}] {side_label} BUY order filled!")
                pos = get_token_position(client, token_id)
                if pos > Decimal("0.01"):
                    sell_at_market(client, token_id, pos, f"{self.label}/{side_label}")
                else:
                    log.warning(f"[{self.label}] {side_label} fill detected but position is 0 — may already be flat.")
 
                # Clear the order ID regardless
                if side_label == "YES":
                    self.yes_order_id = None
                else:
                    self.no_order_id = None
                any_filled = True

        sell_all_positions(client, [self.yes_token_id, self.no_token_id,])
 
        return any_filled
 
    # ── Main cycle ─────────────────────────────────────────
 
    def cycle(self, client: ClobClient) -> None:
        self._cycle_count += 1

        
 
        # 0. Refresh reward params periodically
        if self._cycle_count == 1 or self._cycle_count % self.PARAM_REFRESH_INTERVAL == 0:
            self.refresh_params()
 
        # 1. Skip expired markets — cancel any resting orders
        if self.is_expired():
            if self.yes_order_id or self.no_order_id:
                log.info(f"[{self.label}] Market expired — cancelling orders.")
                self.cancel_all(client)
                had_fill = self.check_fills_and_flatten(client)
            else:
                log.info(f"[{self.label}] Market expired — skipping.")
            return
        

 
        # 2. Detect fills and flatten any acquired position
        had_fill = self.check_fills_and_flatten(client)
        
 
        # 3. Get current YES mid
        yes_mid = get_midpoint(client, self.yes_token_id)
        if yes_mid is None:
            log.warning(f"[{self.label}] Cannot get midpoint — skipping cycle.")
            return
 
        # 4. Requote if: fill, mid drift, or either order is missing
        yes_alive = self.yes_order_id is not None
        no_alive  = self.no_order_id  is not None
        drifted   = self.mid_has_drifted(yes_mid)
 
        if had_fill or drifted or not yes_alive or not no_alive:
            reasons = []
            if had_fill:       reasons.append("fill")
            if drifted:        reasons.append(f"mid drift {self.quoted_yes_mid}→{yes_mid}")
            if not yes_alive:  reasons.append("YES missing")
            if not no_alive:   reasons.append("NO missing")
            log.info(f"[{self.label}] Requoting ({', '.join(reasons)})")
            self.quote(client, yes_mid)
 
        else:
            no_mid = (Decimal("1") - yes_mid).quantize(TICK)
            log.info(
                f"[{self.label}] ✓ Healthy | "
                f"YES mid={yes_mid:.4f} | NO mid={no_mid:.4f} | "
                f"spread=±{self.max_spread} | daily=${self.daily_rate}"
            )

## Main

In [11]:
def run(client: ClobClient, market_configs: list) -> None:
    makers = [MarketMaker(cfg) for cfg in market_configs]
    log.info(f"LP rewards bot started — {len(makers)} market(s).")
    log.info(f"Poll: {POLL_INTERVAL}s | Requote threshold: ±{REQUOTE_THRESHOLD} | Buffer: {SPREAD_BUFFER}")
 
    while True:
        for maker in makers:
            try:
                maker.cycle(client)
            except Exception as e:
                log.error(f"[{maker.label}] Unhandled error: {e}", exc_info=True)
        time.sleep(POLL_INTERVAL)
 
 
def shutdown(client: ClobClient, makers: list) -> None:
    log.info("Shutting down — cancelling all resting orders...")
    for maker in makers:
        maker.cancel_all(client)
    log.info("Shutdown complete.")

# ENTRY POINT

In [12]:
if __name__ == "__main__":
    required_env = [
        "POLYMARKET_API_KEY", "POLYMARKET_API_SECRET", "POLYMARKET_API_PASSPHRASE",
        "POLYMARKET_PRIVATE_KEY", "POLYMARKET_FUNDER",
    ]
    missing = [k for k in required_env if not os.getenv(k)]
    if missing:
        raise EnvironmentError(f"Missing env vars: {', '.join(missing)}")
 
    # Resolve all slugs → full market configs before starting
    log.info("Fetching market configs from Polymarket API...")
    resolved = []
    for m in MARKETS:
        cfg = fetch_market_config(m["slug"], m["label"])
        if cfg is None:
            log.error(f"Could not resolve market: {m['label']} ({m['slug']}) — skipping.")
        elif datetime.now(timezone.utc) >= cfg["end_date"]:
            log.warning(f"Market already expired: {m['label']} — skipping.")
        else:
            resolved.append(cfg)
 
    if not resolved:
        raise RuntimeError("No valid markets to farm. Check your slugs.")
 
    log.info(f"Successfully resolved {len(resolved)} market(s).")
 
    client = build_client()
    makers = [MarketMaker(cfg) for cfg in resolved]
    try:
        run(client, resolved)
    except KeyboardInterrupt:
        shutdown(client, makers)

2026-04-01 05:05:40,490 [INFO] Fetching market configs from Polymarket API...
2026-04-01 05:05:41,085 [INFO] [Seoul Temp] Market config fetched:  YES token: 4666085468268251...  NO  token: 8340982340984370...  End date:  2026-04-01  Max spread: 0.045 (4.5%) | Min size: 50 | Daily rate: $94
2026-04-01 05:05:41,089 [INFO] Successfully resolved 1 market(s).
2026-04-01 05:05:41,132 [INFO] LP rewards bot started — 1 market(s).
2026-04-01 05:05:41,133 [INFO] Poll: 10s | Requote threshold: ±0.005 | Buffer: 0.001
2026-04-01 05:05:41,485 [INFO] [Seoul Temp] Market config fetched:  YES token: 4666085468268251...  NO  token: 8340982340984370...  End date:  2026-04-01  Max spread: 0.045 (4.5%) | Min size: 50 | Daily rate: $94
2026-04-01 05:05:41,487 [INFO] [Seoul Temp] Params refreshed — max_spread=0.03999999999999999989591659144 | min_size=50 | daily_rate=$94
2026-04-01 05:05:42,011 [INFO] HTTP Request: GET https://clob.polymarket.com/book?token_id=466608546826825111509484861968266818174897687735

Minimum Balance required for one event

$$(1 - 2 \times spread) \times min\_size$$

Assume spread = 0.05, min_size = 50

minimum balance required for an event = $45 USDC

In [13]:
yes_token

NameError: name 'yes_token' is not defined

In [15]:
label        = MARKETS[0]["label"]
slug         = MARKETS[0]["slug"]

def fetch_market_config(slug: str, label: str) -> dict | None:
    """
    Fetch all market parameters from the Polymarket gamma API using just the slug.
    Returns a fully populated market config dict, or None on failure.
 
    Fetches:
      - yes_token_id, no_token_id
      - end_date
      - max_spread (converts from % to decimal, e.g. 4.5 -> 0.045)
      - min_size
      - daily_rate
    """
    try:
        resp = requests.get(
            "https://gamma-api.polymarket.com/markets",
            params={"slug": slug},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json()
        if not results:
            log.error(f"[{label}] No market found for slug: {slug}")
            return None
        data = results[0]
 
        # Parse token IDs
        import json
        token_ids = json.loads(data.get("clobTokenIds", "[]"))
        if len(token_ids) < 2:
            log.error(f"[{label}] Expected 2 token IDs, got: {token_ids}")
            return None
 
        # Determine which is YES and which is NO from outcomes
        outcomes = json.loads(data.get("outcomes", '["Yes", "No"]'))
        if outcomes[0].lower() == "yes":
            yes_token_id, no_token_id = token_ids[0], token_ids[1]
        else:
            yes_token_id, no_token_id = token_ids[1], token_ids[0]
 
        # Parse end date
        end_date_str = data.get("endDate", "")
        end_date = datetime.fromisoformat(end_date_str.replace("Z", "+00:00"))
 
        # Reward params
        max_spread_pct = Decimal(str(data.get("rewardsMaxSpread", 5)))
        max_spread     = max_spread_pct / Decimal("100")
        min_size       = Decimal(str(data.get("rewardsMinSize", 50)))
        rewards        = data.get("clobRewards", [{}])[0]
        daily_rate     = Decimal(str(rewards.get("rewardsDailyRate", 0)))
 
        log.info(
            f"[{label}] Market config fetched:"
            f"  YES token: {yes_token_id}"
            f"  NO  token: {no_token_id}"
            f"  End date:  {end_date.date()}"
            f"  Max spread: {max_spread} ({max_spread_pct}%) | "
            f"Min size: {min_size} | Daily rate: ${daily_rate}"
        )
 
        return {
            "label":        label,
            "slug":         slug,
            "yes_token_id": yes_token_id,
            "no_token_id":  no_token_id,
            "end_date":     end_date,
            "max_spread":   max_spread,
            "min_size":     min_size,
            "daily_rate":   daily_rate,
        }
 
    except Exception as e:
        log.error(f"[{label}] fetch_market_config failed: {e}")
        return None
    
def refresh_params(label, slug):
        """Re-fetch all market params from API in case rewards program changed."""
        updated = fetch_market_config(slug, label)
        return updated
updated = refresh_params(label, slug)
        

2026-04-01 09:04:06,279 [INFO] [Seoul Temp] Market config fetched:  YES token: 46660854682682511150948486196826681817489768773578229597926954864279355279449  NO  token: 83409823409843707832955030078212992299246843736648083707916450918424128463271  End date:  2026-04-01  Max spread: 0.045 (4.5%) | Min size: 50 | Daily rate: $33


In [19]:
import logging
from decimal import Decimal
import time
from py_clob_client.clob_types import OrderArgs, BalanceAllowanceParams, AssetType


SIGNATURE_TYPE = 1  # change to match your setup
auth_client = build_client()
sell_all_positions(auth_client, [
    "46660854682682511150948486196826681817489768773578229597926954864279355279449",  # YES
    "83409823409843707832955030078212992299246843736648083707916450918424128463271",   # NO
    # add more token IDs as needed
])

2026-04-01 09:07:22,043 [INFO] Checking position for token 4666085468268251...
2026-04-01 09:07:22,504 [INFO] HTTP Request: GET https://clob.polymarket.com/balance-allowance?asset_type=CONDITIONAL&token_id=46660854682682511150948486196826681817489768773578229597926954864279355279449&signature_type=1 "HTTP/2 200 OK"
2026-04-01 09:07:22,507 [WARNING]   Attempt 1/10 — holding 49.9942 shares. Selling...
2026-04-01 09:07:22,889 [INFO] HTTP Request: GET https://clob.polymarket.com/tick-size?token_id=46660854682682511150948486196826681817489768773578229597926954864279355279449 "HTTP/2 200 OK"
2026-04-01 09:07:22,960 [INFO] HTTP Request: GET https://clob.polymarket.com/neg-risk?token_id=46660854682682511150948486196826681817489768773578229597926954864279355279449 "HTTP/2 200 OK"
2026-04-01 09:07:23,272 [INFO] HTTP Request: GET https://clob.polymarket.com/fee-rate?token_id=46660854682682511150948486196826681817489768773578229597926954864279355279449 "HTTP/2 200 OK"
2026-04-01 09:07:23,611 [INFO